# OCR 대조 시트 점검
합성 PDF와 OCR 정보를 만들고 대조 시트를 생성합니다.

In [ ]:
import csv, json, tempfile
from pathlib import Path
import pymupdf
from payment_card.file_pipeline.extraction.ocr_review_sheet import generate_review_html
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())
work = Path(tempfile.mkdtemp(prefix='ocr-review-check-'))
pdf = work / 'synthetic.pdf'
doc = pymupdf.open(); page = doc.new_page(); page.insert_text((72, 72), 'Synthetic 10,000 won 5 percent'); doc.save(pdf); doc.close()
ocr = work / 'ocr'; ocr.mkdir()
(ocr / 'demo.json').write_text(json.dumps({'pages':[{'page_no':1,'text':'혜택 10,000원 5%'}]}, ensure_ascii=False), encoding='utf-8')
(ocr / 'demo.ocr.json').write_text(json.dumps({'pages':[{'page_no':1,'mean_confidence':75,'words':[{'text':'혜택','confidence':95},{'text':'10,000원','confidence':45},{'text':'5%','confidence':85}]}]}, ensure_ascii=False), encoding='utf-8')
index = work / 'documents.csv'
index.write_text('doc_id,file_path,doc_role,pages\ndemo,synthetic.pdf,synthetic,1\n', encoding='utf-8')
output = work / 'review.html'
generate_review_html(ocr, index, work, output, priority=None)
print({'output': str(output), 'bytes': output.stat().st_size, 'pages': 1})

## 결과 표시
아래 셀에서 생성 파일을 확인합니다.

In [ ]:
from IPython.display import display, HTML
display(HTML('<p>합성 입력으로 HTML 생성 완료: ' + str(output) + '</p>'))